# Phase 1 — EDA Exploratory Data Analysis (Analyse Exploratoire des Données) et préparation des données (FD001)

Objectif de cette première étape : charger `train_FD001.txt` et identifier les capteurs à variance nulle (non informatifs), avant de passer à la normalisation puis au lissage.

Référence colonnes (`data/readme.txt`) : `unit_number`, `time_cycles`, `op_setting_1..3`, `sensor_1..21` (26 colonnes, pas de header dans le fichier source).

In [ ]:
import pandas as pd  # pour charger et manipuler les données sous forme de tableau
import numpy as np  # fonctions numériques utilitaires (pas encore utilisé, mais import standard)

# Noms des 26 colonnes du fichier brut NASA (voir data/readme.txt) - les fichiers texte
# n'ont pas de ligne d'en-tête, donc on doit fournir les noms de colonnes nous-mêmes
COLUMNS = [
    "unit_number",   # identifiant du moteur : 1 à 100 pour FD001 train
    "time_cycles",   # numéro du cycle pour ce moteur, commence à 1
    "op_setting_1",  # réglage opérationnel 1
    "op_setting_2",  # réglage opérationnel 2
    "op_setting_3",  # réglage opérationnel 3
    "sensor_1",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_5",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_16",
    "sensor_17",
    "sensor_18",
    "sensor_19",
    "sensor_20",
    "sensor_21",  # 21 mesures de capteurs au total
]

DATA_DIR = "../data"  # chemin relatif depuis notebooks/ vers le dossier data


def load_fd001(split):
    """
    Charge le fichier train ou test de FD001 dans un DataFrame pandas.

    Parameters
    ----------
    split : str
        "train" ou "test" - sélectionne quel fichier brut charger.

    Returns
    -------
    pandas.DataFrame
        Une ligne par (moteur, cycle), avec les 26 colonnes nommées.
    """
    path = f"{DATA_DIR}/{split}_FD001.txt"  # construit le chemin complet du fichier
    df = pd.read_csv(
        path,
        sep=r"\s+",     # le fichier brut est séparé par des espaces (largeur variable), pas des virgules
        header=None,    # le fichier brut n'a pas de ligne d'en-tête
        names=COLUMNS,  # donc on fournit les noms de colonnes nous-mêmes
    )
    return df  # renvoie le tableau chargé


train = load_fd001("train")  # charge le fichier d'entraînement (100 moteurs, jusqu'à la panne)
train.shape  # vérifie la taille : doit être (20631, 26)

In [ ]:
# Affiche les 5 premières lignes pour vérifier visuellement que les colonnes et valeurs sont correctes
train.head()

## Capteurs à variance nulle

Un capteur constant sur l'ensemble des moteurs ne peut pas être corrélé à la dégradation (il ne varie jamais) : on le retire.

**Attention à la précision flottante** : un test naïf `variance == 0` peut louper un capteur réellement constant si le calcul de variance introduit une erreur d'arrondi (observé ici sur `sensor_5` et `sensor_16` : valeur unique, mais variance calculée ≈ 1e-29 / 1e-35 au lieu de 0 exact). Le critère fiable est le nombre de valeurs distinctes (`nunique() == 1`), pas la variance.

In [ ]:
# Construit la liste des noms de colonnes capteurs (sensor_1 à sensor_21) à partir de COLUMNS,
# avec une boucle simple plutôt qu'une comprehension compacte - plus facile à lire ligne par ligne
sensor_cols = []  # on part d'une liste vide
for column_name in COLUMNS:  # on parcourt chaque nom de colonne, dans l'ordre
    if column_name.startswith("sensor_"):  # on ne garde que les colonnes capteurs
        sensor_cols.append(column_name)  # on l'ajoute à notre liste

variance_per_sensor = train[sensor_cols].var()  # variance de chaque capteur, sur tous moteurs/cycles
unique_values_per_sensor = train[sensor_cols].nunique()  # nombre de valeurs distinctes par capteur

# Met les deux statistiques côte à côte dans un seul tableau, pour pouvoir les comparer directement
sensor_summary = pd.DataFrame({
    "variance": variance_per_sensor,
    "n_unique": unique_values_per_sensor,
})
sensor_summary = sensor_summary.sort_values("n_unique")  # les capteurs à 1 seule valeur apparaissent en premier

sensor_summary  # affiche le tableau

In [ ]:
# Un capteur est réellement constant s'il ne prend qu'UNE seule valeur distincte (n_unique == 1).
# On utilise n_unique ici, pas variance == 0, à cause du problème d'arrondi flottant expliqué
# ci-dessus (sensor_5 et sensor_16 sont constants mais leur variance n'est pas exactement 0)
constant_sensors = []  # on part d'une liste vide
for sensor_name in sensor_summary.index:  # on parcourt chaque capteur
    n_unique = sensor_summary.loc[sensor_name, "n_unique"]  # on récupère son nombre de valeurs distinctes
    if n_unique == 1:  # s'il ne prend jamais plus d'une valeur
        constant_sensors.append(sensor_name)  # il est constant : on l'ajoute à la liste

constant_sensors  # affiche la liste finale des capteurs constants à retirer